In [8]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [9]:
save_dir = "../../output/tables"


In [10]:
for p in df.columns:
    print(p)

text
holistic_essay_score
prompt_name
gender_F
gender_M
grade_level_6.0
grade_level_8.0
grade_level_9.0
grade_level_10.0
grade_level_11.0
grade_level_12.0
race_ethnicity_American Indian/Alaskan Native
race_ethnicity_Asian/Pacific Islander
race_ethnicity_Black/African American
race_ethnicity_Hispanic/Latino
race_ethnicity_Two or more races/Other
race_ethnicity_White
economically_disadvantaged_0
economically_disadvantaged_1
taaled_basic_ntokens
taaled_basic_ntypes
taaled_basic_ncontent_tokens
taaled_basic_ncontent_types
taaled_basic_nfunction_tokens
taaled_basic_nfunction_types
taaled_lexical_density_types
taaled_lexical_density_tokens
taaled_simple_ttr_aw
taaled_simple_ttr_cw
taaled_simple_ttr_fw
taaled_root_ttr_aw
taaled_root_ttr_cw
taaled_root_ttr_fw
taaled_log_ttr_aw
taaled_log_ttr_cw
taaled_log_ttr_fw
taaled_maas_ttr_aw
taaled_maas_ttr_cw
taaled_maas_ttr_fw
taaled_mattr50_aw
taaled_mattr50_cw
taaled_mattr50_fw
taaled_msttr50_aw
taaled_msttr50_cw
taaled_msttr50_fw
taaled_hdd42_aw
taa

In [11]:
# Load Data
df = pd.read_csv("../../data/full/data_full.csv")

# --- Preprocessing ---

# 1. SES: Low vs High
# economically_disadvantaged_1 = Low SES (Free/Reduced Lunch)
# economically_disadvantaged_0 = High SES (Likely Paid Lunch/Not Disadvantaged)
df['ses_group'] = df['economically_disadvantaged_1'].apply(lambda x: 'Low SES' if x == 1 else 'High SES')

# 2. Race: White vs Non-White
df['race_group'] = df['race_ethnicity_White'].apply(lambda x: 'White' if x == 1 else 'Non-White')

# 3. Gender
# Assuming gender_F is reliable for Female.
def get_gender(row):
    if row['gender_F'] == 1:
        return 'Female'
    else:
        return 'Male'
        
df['gender_group'] = df.apply(get_gender, axis=1)

# --- Selection of Metrics ---
# We use existing columns from the dataset instead of calculating new ones.

# Dictionary mapping column names to readable labels/descriptions
metric_descriptions = {
    'holistic_essay_score': 'Holistic Essay Score (Outcome)',
    'gender_F': 'Female',
    'race_ethnicity_White': 'White',
    'grade_level_6.0':"6th Grade",
    'grade_level_8.0':"8th Grade",
    'grade_level_9.0':"9th Grade",
    'grade_level_10.0':"10th Grade",
    'grade_level_11.0':"11th Grade",
    'grade_level_11.0':"12th Grade",
    'taassc_nwords': 'Word Count (Volume)',
    'taassc_wrd_length': 'Avg Word Length (Lexical Sophistication)',
    'taassc_mlc': 'Mean Length of Clause (Syntactic Complexity)',
    'taassc_mltu': 'Mean Length of T-Unit (Sentence Complexity)',
    'taassc_mean_verbal_deps': 'Mean Verbal Dependencies',
    'taassc_infinitive_prop': 'Proportion of Infinitives',
    'taassc_nonfinite_prop': 'Proportion of Non-Finite Clauses',
    "taaled_lexical_density_tokens": "Lexical density (tokens)",
    "taaled_mtld_original_aw": "Lexical diversity (MTLD; all words)",
    "taaco_lemma_mattr": "Lexical diversity (MATTR; lemmas)",
    "taaco_adjacent_overlap_cw_sent": "Content-word overlap (adjacent sentences)",
    "taaco_word2vec_1_all_sent": "Semantic cohesion (Word2Vec; adjacent sentences)",
    "taaco_basic_connectives": "Connectives (basic)",
    "taaco_reason_and_purpose": "Causal connectives (reason & purpose)",
    "taassc_nominalization": "Nominalizations",
    "taaco_pronoun_noun_ratio": "Pronoun-to-noun ratio",
}

target_metrics = list(metric_descriptions.keys())

print("Data loaded and groups defined. Using metrics:")
for m, desc in metric_descriptions.items():
    print(f" - {m}: {desc}")

Data loaded and groups defined. Using metrics:
 - holistic_essay_score: Holistic Essay Score (Outcome)
 - gender_F: Female
 - race_ethnicity_White: White
 - grade_level_6.0: 6th Grade
 - grade_level_8.0: 8th Grade
 - grade_level_9.0: 9th Grade
 - grade_level_10.0: 10th Grade
 - grade_level_11.0: 12th Grade
 - taassc_nwords: Word Count (Volume)
 - taassc_wrd_length: Avg Word Length (Lexical Sophistication)
 - taassc_mlc: Mean Length of Clause (Syntactic Complexity)
 - taassc_mltu: Mean Length of T-Unit (Sentence Complexity)
 - taassc_mean_verbal_deps: Mean Verbal Dependencies
 - taassc_infinitive_prop: Proportion of Infinitives
 - taassc_nonfinite_prop: Proportion of Non-Finite Clauses
 - taaled_lexical_density_tokens: Lexical density (tokens)
 - taaled_mtld_original_aw: Lexical diversity (MTLD; all words)
 - taaco_lemma_mattr: Lexical diversity (MATTR; lemmas)
 - taaco_adjacent_overlap_cw_sent: Content-word overlap (adjacent sentences)
 - taaco_word2vec_1_all_sent: Semantic cohesion (W

In [12]:
# def get_group_stats(df, group_col, group_label_map, metrics):
#     results = {}
    
#     # group_label_map: {'GroupValue': 'DisplayName'}
#     # e.g., {'Low SES': 'Low SES', 'High SES': 'High SES'}
    
#     for group_val, display_name in group_label_map.items():
#         sub_df = df[df[group_col] == group_val]
#         n = len(sub_df)
#         stats = sub_df[metrics].mean().to_dict()
        
#         # Add relevant metadata
#         col_data = {'N': n}
#         for m in metrics:
#             sd_val = sub_df[metrics].std()[m]
#             col_data[f"{m}_mean"] = stats[m]
#             col_data[f"{m}_sd"] = sd_val
#             col_data[f"{m}_se"] = sd_val / np.sqrt(n) if n > 0 else np.nan
            
#         results[display_name] = col_data
        
#     return pd.DataFrame(results)

# # 1. Full Sample
# full_stats = pd.DataFrame(index=['N'] + [f"{m}_{s}" for m in target_metrics for s in ['mean', 'sd', 'se']])
# full_stats['Full Sample'] = pd.Series({'N': len(df)})
# for m in target_metrics:
#     sd_val = df[m].std()
#     full_stats.loc[f"{m}_mean", 'Full Sample'] = df[m].mean()
#     full_stats.loc[f"{m}_sd", 'Full Sample'] = sd_val
#     full_stats.loc[f"{m}_se", 'Full Sample'] = sd_val / np.sqrt(len(df))

# # 2. SES Breakdown
# ses_stats = get_group_stats(df, 'ses_group', {'Low SES': 'Low SES', 'High SES': 'High SES'}, target_metrics)

# # Combine all
# final_table = pd.concat([full_stats, ses_stats], axis=1)

# # Calculate Gaps
# gap_col = pd.Series(index=final_table.index, dtype=float, name='Gap SES (High-Low)')
# gap_col.loc['N'] = np.nan
# for m in target_metrics:
#     gap_col.loc[f"{m}_mean"] = final_table.loc[f"{m}_mean", 'High SES'] - final_table.loc[f"{m}_mean", 'Low SES']
#     gap_col.loc[f"{m}_sd"] = np.nan
#     se_high = final_table.loc[f"{m}_se", 'High SES']
#     se_low = final_table.loc[f"{m}_se", 'Low SES']
#     if pd.notna(se_high) and pd.notna(se_low):
#         gap_col.loc[f"{m}_se"] = np.sqrt(se_high**2 + se_low**2)
#     else:
#         gap_col.loc[f"{m}_se"] = np.nan

# final_table['Gap SES (High-Low)'] = gap_col

# # Reorder for display
# cols_order = [
#     'Full Sample', 
#     'Low SES', 'High SES', 'Gap SES (High-Low)',
#     # 'Non-White', 'White', 'Gap Race (White-NonWhite)',
#     # 'Male', 'Female', 'Gap Gender (F-M)'
# ]
# final_table = final_table[cols_order]

# def make_mean_se_table(raw_table, metrics, col_order, metric_descriptions):
#     """Format each metric as mean with standard error beneath for display/LaTeX."""
#     formatted = pd.DataFrame(index=[metric_descriptions.get(m, m) for m in metrics], columns=col_order)
#     for col in col_order:
#         for m in metrics:
#             mean_val = raw_table.loc[f"{m}_mean", col]
#             se_val = raw_table.loc[f"{m}_se", col]
#             label = metric_descriptions.get(m, m)
#             if pd.isna(mean_val):
#                 formatted.loc[label, col] = ""
#                 continue
#             if pd.isna(se_val):
#                 formatted.loc[label, col] = f"{mean_val:.3f}"
#             else:
#                 formatted.loc[label, col] = f"\shortstack{{{mean_val:.3f} \ ({se_val:.3f})}}"
#     return formatted

# formatted_table = make_mean_se_table(final_table, target_metrics, cols_order, metric_descriptions)

# # Add Ns as the first row for quick reference
# formatted_table.loc['N'] = [
#     int(final_table.loc['N', col]) if pd.notna(final_table.loc['N', col]) else ''
#     for col in cols_order
# ]
# formatted_table = formatted_table.loc[['N'] + [metric_descriptions[m] for m in target_metrics]]

# # Display
# formatted_table


In [13]:
# # Export numeric table to CSV
# final_table.to_csv("descriptive_statistics_summary.csv")

# # Export formatted mean/SE table to LaTeX
# latex_path = "descriptive_statistics_summary.tex"
# formatted_table.to_latex(latex_path, escape=False)

# # Print descriptions again for context in logical place
# print("Column Descriptions included in analysis:")
# for m, desc in metric_descriptions.items():
#     print(f" - {m}: {desc}")
    
# print("Table saved to descriptive_statistics_summary.csv")
# print(f"LaTeX table saved to {latex_path}")


In [14]:
import numpy as np
import pandas as pd


def mean_se(sub_df, metrics):
    n = len(sub_df)
    mean = sub_df[metrics].mean()
    sd = sub_df[metrics].std(ddof=1)
    se = sd / np.sqrt(n)
    return n, mean, sd, se

def fmt_cell(mean, se, digits=3):
    return rf"\shortstack{{{mean:.{digits}f} \\ ({se:.{digits}f})}}"

# --- compute stats ---
nF, meanF, sdF, seF = mean_se(df, target_metrics)

low_df  = df[df["ses_group"] == "Low SES"]
high_df = df[df["ses_group"] == "High SES"]

nL, meanL, sdL, seL = mean_se(low_df, target_metrics)
nH, meanH, sdH, seH = mean_se(high_df, target_metrics)

# Gap = High - Low
gap_mean = meanH - meanL
gap_se = np.sqrt((sdH**2)/nH + (sdL**2)/nL)  # diff-in-means SE

# --- build table with pretty row labels ---
row_labels = ["N"] + [metric_descriptions.get(m, m) for m in target_metrics]
final_table = pd.DataFrame(index=row_labels)

final_table["Full Sample"] = [str(nF)] + [fmt_cell(meanF[m], seF[m]) for m in target_metrics]
final_table["Low SES"]     = [str(nL)] + [fmt_cell(meanL[m], seL[m]) for m in target_metrics]
final_table["High SES"]    = [str(nH)] + [fmt_cell(meanH[m], seH[m]) for m in target_metrics]
final_table["Gap SES (High-Low)"] = [""] + [fmt_cell(gap_mean[m], gap_se[m]) for m in target_metrics]

# optional column order
cols_order = ["Full Sample", "Low SES", "High SES", "Gap SES (High-Low)"]
final_table = final_table[cols_order]

# --- export to LaTeX ---
latex_str = final_table.to_latex(
    escape=False,  # keep \shortstack
    index=True
)

with open(save_dir + "table.tex", "w") as f:
    f.write(latex_str)

final_table

,Full Sample,Low SES,High SES,Gap SES (High-Low)
N,20759,9643,11116,
Holistic Essay Score (Outcome),\shortstack{3.337 \\ (0.008)},\shortstack{2.979 \\ (0.011)},\shortstack{3.648 \\ (0.011)},\shortstack{0.669 \\ (0.016)}
Female,\shortstack{0.512 \\ (0.003)},\shortstack{0.526 \\ (0.005)},\shortstack{0.500 \\ (0.005)},\shortstack{-0.026 \\ (0.007)}
White,\shortstack{0.452 \\ (0.003)},\shortstack{0.247 \\ (0.004)},\shortstack{0.630 \\ (0.005)},\shortstack{0.384 \\ (0.006)}
6th Grade,\shortstack{0.066 \\ (0.002)},\shortstack{0.073 \\ (0.003)},\shortstack{0.060 \\ (0.002)},\shortstack{-0.012 \\ (0.003)}
8th Grade,\shortstack{0.461 \\ (0.003)},\shortstack{0.421 \\ (0.005)},\shortstack{0.495 \\ (0.005)},\shortstack{0.075 \\ (0.007)}
9th Grade,\shortstack{0.001 \\ (0.000)},\shortstack{0.001 \\ (0.000)},\shortstack{0.001 \\ (0.000)},\shortstack{0.000 \\ (0.000)}
10th Grade,\shortstack{0.304 \\ (0.003)},\shortstack{0.376 \\ (0.005)},\shortstack{0.242 \\ (0.004)},\shortstack{-0.134 \\ (0.006)}
12th Grade,\shortstack{0.149 \\ (0.002)},\shortstack{0.101 \\ (0.003)},\shortstack{0.190 \\ (0.004)},\shortstack{0.089 \\ (0.005)}
Word Count (Volume),\shortstack{412.621 \\ (1.348)},\shortstack{368.372 \\ (1.756)},\shortstack{451.006 \\ (1.932)},\shortstack{82.635 \\ (2.610)}
